In [7]:
# =============================================================================
# 🍇 GRAPE — FINAL MASTER IMAGE TESTING PIPELINE
# =============================================================================
# Standard: Dual-Pool Evaluation (Internal vs True External Real-World)
# Models Tested: 6th Trained_Model/grape_cnn_best.keras
#
# INTERNAL TESTING:
# 30 Images (10 per class) from the known dataset pool (Final_Combined / validation split).
#
# EXTERNAL REAL-WORLD TESTING:
# 30 Completely NEW images (10 per class) from the open internet / real-time camera clicks,
# stored in 7th Test_Images/Grape/External real world data/{Healthy, Black_Rot, Leaf_Blight}.
# =============================================================================

from pathlib import Path

# 1. GOOGLE DRIVE MOUNT
# =============================================================================
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/Plant Disease Detection (Computer Vision)")
except Exception:
    PROJECT_ROOT = Path(r"G:/My Drive/Plant Disease Detection (Computer Vision)")

if not PROJECT_ROOT.is_dir():
    PROJECT_ROOT = Path(r"G:\My Drive\Plant Disease Detection (Computer Vision)")

print("=" * 90)
print("🍇 GRAPE — FINAL MASTER IMAGE TESTING PIPELINE")
print("=" * 90)
print(f"Project root: {PROJECT_ROOT}")

# 2. IMPORTS & SYSTEM CONFIGURATION
# =============================================================================
import os
import json
import random
import shutil
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.models import load_model
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# 3. PATH CONFIGURATION
# =============================================================================
CLASS_NAMES = ["Healthy", "Black_Rot", "Leaf_Blight"]
CLASS_TO_ID = {name: idx for idx, name in enumerate(CLASS_NAMES)}
NUM_CLASSES = len(CLASS_NAMES)
IMAGES_PER_CLASS = 10
TOTAL_POOL_IMAGES = 30
VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}

MODEL_PATH = PROJECT_ROOT / "6th Trained_Model" / "grape_cnn_best.keras"
NPZ_PATH = PROJECT_ROOT / "3rd Preprocessing" / "grape_processed_data.npz"
INTERNAL_ROOT = PROJECT_ROOT / "1st Raw_Data" / "Grape" / "Final_Combined"

EXTERNAL_ROOT = PROJECT_ROOT / "7th Test_Images" / "Grape" / "External real world data"
# Fallback to Grape_RealWorld if External real world data is empty
if not EXTERNAL_ROOT.is_dir() or sum(len(list((EXTERNAL_ROOT / c).glob("*"))) for c in CLASS_NAMES if (EXTERNAL_ROOT / c).is_dir()) == 0:
    alt_ext = PROJECT_ROOT / "7th Test_Images" / "Grape_RealWorld"
    if alt_ext.is_dir() and sum(len(list((alt_ext / c).glob("*"))) for c in CLASS_NAMES if (alt_ext / c).is_dir()) >= 30:
        EXTERNAL_ROOT = alt_ext

RESULTS_DIR = PROJECT_ROOT / "7th Test_Images" / "Grape" / "Testing_Results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / "Internal_Testing").mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / "External_Testing").mkdir(parents=True, exist_ok=True)

print(f"Model path     : {MODEL_PATH}")
print(f"NPZ path       : {NPZ_PATH}")
print(f"Internal root  : {INTERNAL_ROOT}")
print(f"External root  : {EXTERNAL_ROOT}")
print(f"Results dir    : {RESULTS_DIR}")

# 4. LOAD TRAINED MODEL
# =============================================================================
print("\n" + "=" * 90)
print("LOADING TRAINED MODEL")
print("=" * 90)
if not MODEL_PATH.is_file():
    raise FileNotFoundError(f"Trained model not found at: {MODEL_PATH}")

model = load_model(MODEL_PATH)
print("✅ Trained model loaded successfully.")
print(f"Model input shape : {model.input_shape}")
print(f"Model output shape: {model.output_shape}")

# 5. BUILD INTERNAL TESTING POOL (30 Images: 10/class from Known Data)
# =============================================================================
print("\n" + "=" * 90)
print("BUILDING INTERNAL TESTING POOL (From Known Project Data)")
print("=" * 90)

internal_records = []

# Deterministic selection from Final_Combined
for class_name in CLASS_NAMES:
    class_folder = INTERNAL_ROOT / class_name
    if not class_folder.is_dir():
        raise FileNotFoundError(f"Internal class folder missing: {class_folder}")

    files = sorted([
        f for f in class_folder.iterdir()
        if f.is_file() and f.suffix.lower() in VALID_EXTENSIONS
    ])

    # Deterministic stride to pick representative images
    rng = random.Random(SEED)
    sampled = rng.sample(files, IMAGES_PER_CLASS)

    for f in sorted(sampled):
        internal_records.append({
            "testing_type": "Internal",
            "filepath": str(f),
            "filename": f.name,
            "true_class": class_name,
            "true_label": CLASS_TO_ID[class_name],
            "source": "Final_Combined_Validation_Pool"
        })
    print(f"  Internal {class_name:<15}: {len(sampled)} images selected")

internal_df = pd.DataFrame(internal_records)
assert len(internal_df) == 30, f"Internal pool must be 30 images, got {len(internal_df)}"
print("✅ Internal testing pool ready: Exactly 30 images (10 per class).")

# 6. BUILD EXTERNAL REAL-WORLD TESTING POOL (30 Images: 10/class from Internet/Camera)
# =============================================================================
print("\n" + "=" * 90)
print("BUILDING EXTERNAL REAL-WORLD TESTING POOL (From Internet / New Real-World)")
print("=" * 90)

external_records = []

for class_name in CLASS_NAMES:
    class_folder = EXTERNAL_ROOT / class_name
    if not class_folder.is_dir():
        raise FileNotFoundError(f"External class folder missing: {class_folder}")

    files = sorted([
        f for f in class_folder.iterdir()
        if f.is_file() and f.suffix.lower() in VALID_EXTENSIONS
    ])

    if len(files) < IMAGES_PER_CLASS:
        raise RuntimeError(
            f"STOP: External class '{class_name}' requires at least {IMAGES_PER_CLASS} images.\n"
            f"Found only {len(files)} in: {class_folder}\n"
            f"Please place {IMAGES_PER_CLASS} real-world internet/camera images in that folder."
        )

    selected_files = files[:IMAGES_PER_CLASS]
    for f in selected_files:
        external_records.append({
            "testing_type": "External",
            "filepath": str(f),
            "filename": f.name,
            "true_class": class_name,
            "true_label": CLASS_TO_ID[class_name],
            "source": "External_RealWorld_New_Internet"
        })
    print(f"  External {class_name:<15}: {len(selected_files)} new images selected")

external_df = pd.DataFrame(external_records)
assert len(external_df) == 30, f"External pool must be 30 images, got {len(external_df)}"
print("✅ External testing pool ready: Exactly 30 images (10 per class).")

# 7. IMAGE PREPROCESSING & INFERENCE FUNCTION
# =============================================================================
def load_and_preprocess_image(path):
    with Image.open(path) as img:
        img = img.convert("RGB")
        img = img.resize((224, 224), Image.Resampling.BILINEAR)
        arr = np.array(img, dtype=np.uint8)
    return arr

def run_pool_inference(df):
    results = []
    for idx, row in df.iterrows():
        img_arr = load_and_preprocess_image(row["filepath"])
        batch = np.expand_dims(img_arr, axis=0)
        probs = model.predict(batch, verbose=0)[0]
        pred_label = int(np.argmax(probs))
        pred_class = CLASS_NAMES[pred_label]
        conf = float(probs[pred_label])
        correct = (pred_label == row["true_label"])

        results.append({
            "testing_type": row["testing_type"],
            "filename": row["filename"],
            "filepath": row["filepath"],
            "true_class": row["true_class"],
            "true_label": row["true_label"],
            "pred_class": pred_class,
            "pred_label": pred_label,
            "confidence": conf,
            "correct": correct,
            "prob_healthy": float(probs[0]),
            "prob_black_rot": float(probs[1]),
            "prob_leaf_blight": float(probs[2])
        })
    return pd.DataFrame(results)

# 8. RUN TESTING
# =============================================================================
print("\n" + "=" * 90)
print("RUNNING INFERENCE ON INTERNAL & EXTERNAL TEST POOLS")
print("=" * 90)

print("Running Internal testing (30 images)...")
int_results_df = run_pool_inference(internal_df)

print("Running External real-world testing (30 images)...")
ext_results_df = run_pool_inference(external_df)

# Combine results
combined_df = pd.concat([int_results_df, ext_results_df], ignore_index=True)

# 9. PERFORMANCE METRICS & SUMMARY
# =============================================================================
int_acc = int_results_df["correct"].mean() * 100
ext_acc = ext_results_df["correct"].mean() * 100
acc_gap = int_acc - ext_acc

print("\n" + "=" * 90)
print("🍇 GRAPE FINAL IMAGE TESTING BENCHMARK RESULTS")
print("=" * 90)
print(f"Internal Testing Accuracy  : {int_acc:.2f}% ({int_results_df['correct'].sum()}/30 correct)")
print(f"External Testing Accuracy  : {ext_acc:.2f}% ({ext_results_df['correct'].sum()}/30 correct)")
print(f"Internal - External Gap    : {acc_gap:.2f}%")

print("\n--- INTERNAL CLASS BREAKDOWN ---")
for c in CLASS_NAMES:
    sub = int_results_df[int_results_df["true_class"] == c]
    print(f"  {c:<15}: {sub['correct'].mean() * 100:.2f}% ({sub['correct'].sum()}/{len(sub)} correct) | Avg Conf: {sub['confidence'].mean():.4f}")

print("\n--- EXTERNAL REAL-WORLD CLASS BREAKDOWN ---")
for c in CLASS_NAMES:
    sub = ext_results_df[ext_results_df["true_class"] == c]
    print(f"  {c:<15}: {sub['correct'].mean() * 100:.2f}% ({sub['correct'].sum()}/{len(sub)} correct) | Avg Conf: {sub['confidence'].mean():.4f}")

# 10. SAVE DETAILED REPORTS & CSVs
# =============================================================================
int_csv = RESULTS_DIR / "grape_internal_predictions.csv"
ext_csv = RESULTS_DIR / "grape_external_predictions.csv"
comb_csv = RESULTS_DIR / "grape_combined_predictions.csv"
comp_csv = RESULTS_DIR / "grape_internal_vs_external_comparison.csv"
summary_json = RESULTS_DIR / "grape_final_testing_summary.json"

int_results_df.to_csv(int_csv, index=False)
ext_results_df.to_csv(ext_csv, index=False)
combined_df.to_csv(comb_csv, index=False)

comparison_data = [
    {
        "Dataset": "Internal (Known Project Data)",
        "Images": 30,
        "Accuracy (%)": round(int_acc, 2),
        "Healthy Acc (%)": round(int_results_df[int_results_df["true_class"] == "Healthy"]["correct"].mean() * 100, 2),
        "Black_Rot Acc (%)": round(int_results_df[int_results_df["true_class"] == "Black_Rot"]["correct"].mean() * 100, 2),
        "Leaf_Blight Acc (%)": round(int_results_df[int_results_df["true_class"] == "Leaf_Blight"]["correct"].mean() * 100, 2)
    },
    {
        "Dataset": "External (New Internet / Real-World)",
        "Images": 30,
        "Accuracy (%)": round(ext_acc, 2),
        "Healthy Acc (%)": round(ext_results_df[ext_results_df["true_class"] == "Healthy"]["correct"].mean() * 100, 2),
        "Black_Rot Acc (%)": round(ext_results_df[ext_results_df["true_class"] == "Black_Rot"]["correct"].mean() * 100, 2),
        "Leaf_Blight Acc (%)": round(ext_results_df[ext_results_df["true_class"] == "Leaf_Blight"]["correct"].mean() * 100, 2)
    }
]
pd.DataFrame(comparison_data).to_csv(comp_csv, index=False)

summary_dict = {
    "Plant": "Grape",
    "Model": "MobileNetV2",
    "Internal_Accuracy": round(int_acc, 2),
    "External_Accuracy": round(ext_acc, 2),
    "Accuracy_Gap": round(acc_gap, 2),
    "Internal_Correct": int(int_results_df['correct'].sum()),
    "External_Correct": int(ext_results_df['correct'].sum()),
    "Total_Evaluated": 60,
    "Classes": CLASS_NAMES
}

with open(summary_json, "w") as f:
    json.dump(summary_dict, f, indent=2)

print(f"\n✅ All testing reports saved to: {RESULTS_DIR}")

# 11. GENERATE PREDICTION VISUALIZATION GRIDS (Internal & External)
# =============================================================================
def plot_prediction_grid(df, title, save_path):
    fig, axes = plt.subplots(5, 6, figsize=(18, 15))
    axes = axes.flatten()

    for i, (_, row) in enumerate(df.iterrows()):
        ax = axes[i]
        with Image.open(row["filepath"]) as img:
            ax.imshow(img)

        color = "green" if row["correct"] else "red"
        status = "✓" if row["correct"] else "✗"
        title_text = f"{status} Pred: {row['pred_class']}\nTrue: {row['true_class']}\nConf: {row['confidence']*100:.1f}%"
        ax.set_title(title_text, color=color, fontsize=9, fontweight="bold")
        ax.axis("off")

        for spine in ax.spines.values():
            spine.set_color(color)
            spine.set_linewidth(2)
            spine.set_visible(True)

    plt.suptitle(title, fontsize=16, fontweight="bold", y=0.99)
    plt.tight_layout()
    plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.close()
    print(f"✅ Saved visualization grid: {save_path.name}")

int_grid_png = RESULTS_DIR / "grape_internal_predictions_visualization.png"
ext_grid_png = RESULTS_DIR / "grape_external_predictions_visualization.png"

plot_prediction_grid(int_results_df, "🍇 Grape — Internal Testing (30 Images: 10/class from Known Data)", int_grid_png)
plot_prediction_grid(ext_results_df, "🌍 Grape — External Testing (30 Images: 10/class from Internet/Real-World)", ext_grid_png)

print("\n" + "=" * 90)
print("🎉 FINAL MASTER IMAGE TESTING FOR GRAPE COMPLETED SUCCESSFULLY!")
print("=" * 90)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🍇 GRAPE — FINAL MASTER IMAGE TESTING PIPELINE
Project root: /content/drive/MyDrive/Plant Disease Detection (Computer Vision)
Model path     : /content/drive/MyDrive/Plant Disease Detection (Computer Vision)/6th Trained_Model/grape_cnn_best.keras
NPZ path       : /content/drive/MyDrive/Plant Disease Detection (Computer Vision)/3rd Preprocessing/grape_processed_data.npz
Internal root  : /content/drive/MyDrive/Plant Disease Detection (Computer Vision)/1st Raw_Data/Grape/Final_Combined
External root  : /content/drive/MyDrive/Plant Disease Detection (Computer Vision)/7th Test_Images/Grape/External real world data
Results dir    : /content/drive/MyDrive/Plant Disease Detection (Computer Vision)/7th Test_Images/Grape/Testing_Results

LOADING TRAINED MODEL
✅ Trained model loaded successfully.
Model input shape : (None, 224, 224, 3)
Model output shape: (None, 3)

BUIL

/tmp/ipykernel_2397/929795786.py:325: UserWarning: Glyph 127815 (\N{GRAPES}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2397/929795786.py:326: UserWarning: Glyph 127815 (\N{GRAPES}) missing from font(s) DejaVu Sans.
  plt.savefig(save_path, dpi=200, bbox_inches="tight")


✅ Saved visualization grid: grape_internal_predictions_visualization.png


/tmp/ipykernel_2397/929795786.py:325: UserWarning: Glyph 127757 (\N{EARTH GLOBE EUROPE-AFRICA}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2397/929795786.py:326: UserWarning: Glyph 127757 (\N{EARTH GLOBE EUROPE-AFRICA}) missing from font(s) DejaVu Sans.
  plt.savefig(save_path, dpi=200, bbox_inches="tight")


✅ Saved visualization grid: grape_external_predictions_visualization.png

🎉 FINAL MASTER IMAGE TESTING FOR GRAPE COMPLETED SUCCESSFULLY!
